# Stage 2: Document Parsing

Based on the results from `01_raw_pdf_inspection.ipynb`, the San Diego Municipal Code PDFs are born-digital and contain extractable text layers.

This notebook bulk-processes all 245 PDFs across Chapters 11-15, extracting the text using `unstructured` (fast strategy) and saving the results into a structured `.jsonl` file. This preserves essential metadata (like chapter, filename, and element type) required for downstream chunking and accurate citation during RAG retrieval.

In [1]:
import os
import json
from pathlib import Path
from unstructured.partition.pdf import partition_pdf
import warnings
warnings.filterwarnings('ignore')

# Constants
DATA_DIR = Path("../data/raw")
OUTPUT_DIR = Path("../data/processed")
OUTPUT_FILE = OUTPUT_DIR / "san_diego_code_structured.jsonl"
CHAPTERS = ["chapter_11", "chapter_12", "chapter_13", "chapter_14", "chapter_15"]

# Ensure output directory exists
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data Directory: {DATA_DIR.resolve()}")
print(f"Output File: {OUTPUT_FILE.resolve()}")

/Users/Aresh/Desktop/AAI 590/sd-land-use-rag/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/Users/Aresh/Desktop/AAI 590/sd-land-use-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data Directory: /Users/Aresh/Desktop/AAI 590/sd-land-use-rag/data/raw
Output File: /Users/Aresh/Desktop/AAI 590/sd-land-use-rag/data/processed/san_diego_code_structured.jsonl


## 1. Bulk Parse PDFs to JSONL

We iterate through every PDF, partition the elements, and write them sequentially to a JSON Lines (`.jsonl`) file. This streaming approach prevents memory bloat.

In [2]:
def parse_and_save_pdfs():
    total_pdfs = 0
    total_elements = 0
    
    # Open the JSONL file in write mode
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        for chapter in CHAPTERS:
            chapter_path = DATA_DIR / chapter
            pdf_files = list(chapter_path.glob("*.pdf"))
            print(f"Processing {len(pdf_files)} files in {chapter}...")
            
            for pdf_path in pdf_files:
                total_pdfs += 1
                try:
                    # Extract elements using the fast strategy
                    elements = partition_pdf(filename=str(pdf_path), strategy="fast")
                    
                    for el in elements:
                        text = str(el).strip()
                        if not text:
                            continue
                            
                        # Construct the metadata record
                        record = {
                            "chapter": chapter,
                            "filename": pdf_path.name,
                            "element_type": type(el).__name__,
                            "text": text
                        }
                        
                        # Write individual JSON object as a line
                        f.write(json.dumps(record, ensure_ascii=False) + "\n")
                        total_elements += 1
                        
                except Exception as e:
                    print(f"Error processing {pdf_path.name}: {e}")
                    
    print("\n--- Parsing Complete ---")
    print(f"Processed {total_pdfs} PDFs.")
    print(f"Extracted {total_elements} textual elements.")
    print(f"Saved to: {OUTPUT_FILE.name}")

# Execute parsing (This may take a few minutes)
parse_and_save_pdfs()

Processing 10 files in chapter_11...
Processing 49 files in chapter_12...
Processing 22 files in chapter_13...
Processing 108 files in chapter_14...
Processing 56 files in chapter_15...

--- Parsing Complete ---
Processed 245 PDFs.
Extracted 64639 textual elements.
Saved to: san_diego_code_structured.jsonl


## 2. Verification and Sanity Check

Let's load the `.jsonl` file back into memory to confirm the structure, count elements per chapter, and inspect a few samples to ensure we captured section markers (e.g., `§`).

In [3]:
import pandas as pd

print("Loading JSONL into Pandas for verification...")
df = pd.read_json(OUTPUT_FILE, lines=True)

print(f"\nTotal records loaded: {len(df)}")
print("\nBreakdown by Chapter:")
print(df['chapter'].value_counts())

print("\nBreakdown by Element Type:")
print(df['element_type'].value_counts())

print("\n--- Sample Row (Searching for Section Markers '§') ---")
section_samples = df[df['text'].str.contains("§", na=False)]
if not section_samples.empty:
    sample = section_samples.iloc[0]
    print(json.dumps(sample.to_dict(), indent=2, ensure_ascii=False))
else:
    print("WARNING: No section markers '§' found in the extracted text.\nLook at random samples instead:")
    print(df.sample(2).to_dict(orient='records'))

Loading JSONL into Pandas for verification...

Total records loaded: 64639

Breakdown by Chapter:
chapter
chapter_13    21701
chapter_14    21060
chapter_15    13391
chapter_12     5952
chapter_11     2535
Name: count, dtype: int64

Breakdown by Element Type:
element_type
Text             23468
Title            20229
NarrativeText    14017
Header            4725
Footer            1208
ListItem           992
Name: count, dtype: int64

--- Sample Row (Searching for Section Markers '§') ---
{
  "chapter": "chapter_11",
  "filename": "Ch11Art03Division02.pdf",
  "element_type": "Text",
  "text": "§113.0201"
}
